# Step 10: Final robustness checks

Reproduction, random seeds, the time split, Transformer tuning, the punctuation question on other datasets, and the team ensemble. Code: `experiments/14_final_checks/`. Details: `docs/steps/10_final_checks.md`.

In [1]:
# Imports and paths
import warnings; warnings.filterwarnings("ignore")
import json, glob
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
import nbutils as U
F = U.EXP / "14_final_checks/results"
pd.set_option("display.width", 200)

## 10.1 Reproduction of the submitted ensemble
`ensemble_v3.py` rerun from scratch with the pretrained members dropped, compared with the saved run.

In [2]:
# Rerun vs saved results
a = pd.read_csv(U.EXP / "11_full_convergence/results/ensemble_compliant_repro/ensemble_results.csv")
b = pd.read_csv(U.EXP / "11_full_convergence/results/ensemble_compliant_with_scratch/ensemble_results.csv")
m = a.merge(b, on=["set", "method"], suffixes=("_rerun", "_saved"))
print("largest test AUC difference:", f"{(m.test_roc_auc_rerun - m.test_roc_auc_saved).abs().max():.1e}")
U.show(m[m.method.isin(["best_single", "stack_lr", "blend_meta_lr"])][["set", "method", "test_roc_auc_rerun", "test_roc_auc_saved"]])

largest test AUC difference: 3.2e-05


,set,method,test_roc_auc_rerun,test_roc_auc_saved
0,sarc_random,best_single,0.8296,0.8296
3,sarc_random,stack_lr,0.8416,0.8416
7,sarc_random,blend_meta_lr,0.8420,0.8420
10,sarc_subreddit,best_single,0.8226,0.8226
13,sarc_subreddit,stack_lr,0.8335,0.8335
17,sarc_subreddit,blend_meta_lr,0.8340,0.8340


## 10.2 Random seeds (sarc_random)
Three seeds (3244, 3245, 3246) for the four seeded compliant models; the stack is refitted once per seed.

In [3]:
# Mean and spread over seeds
sv = U.read_json(F / "seed_variability.json")
rows = [{"model": k, "mean": v["mean"], "std": v["std"], **{f"seed {s}": a for s, a in v["per_seed"].items()}} for k, v in sv.items()]
U.show(pd.DataFrame(rows))

,model,mean,std,seed 3244,seed 3245,seed 3246
0,lgbm_lrscore_cues_contrast,0.8299,0.0003,0.8296,0.8302,0.8300
1,ebm_full,0.8229,0.0000,0.8229,0.8229,0.8229
2,scratch_pair,0.8237,0.0004,0.8242,0.8235,0.8235
3,scratch_reply,0.8180,0.0007,0.8180,0.8173,0.8187
4,compliant LR stack,0.8417,0.0003,0.8416,0.8420,0.8414


## 10.3 Generalising across time (`sarc_temporal`)
Models refitted on the time split's training years with settings frozen from sarc_random; the test set is later in time.

In [4]:
# Single models on sarc_temporal
ts = pd.read_csv(F / "temporal_single_models.csv")
ts["label"] = [U.info(m)["label"] for m in ts.model]
U.show(ts[["label", "test_auc", "d_vs_baseline", "ci_low", "ci_high"]])

,label,test_auc,d_vs_baseline,ci_low,ci_high
0,LightGBM + LR score + cues + contrast,0.8126,0.0209,0.0199,0.0219
1,Full-data EBM (66 features + text score),0.8054,0.0136,0.0125,0.0147
2,"From-scratch ESIM, reply + parent",0.8050,0.0133,0.0119,0.0146
3,"LR, word + char n-grams",0.8011,0.0093,0.0081,0.0104
4,"From-scratch BiLSTM, reply only",0.8010,0.0092,0.0079,0.0106
5,"LightGBM, + surface (baseline)",0.7918,NaN,NaN,NaN
6,"XGBoost, + surface",0.7906,-0.0011,-0.0016,-0.0006
7,"Logistic regression, + surface",0.7856,-0.0061,-0.0072,-0.0054
8,"LightGBM, word TF-IDF",0.7806,-0.0112,-0.0119,-0.0104
9,"XGBoost, word TF-IDF",0.7798,-0.0119,-0.0127,-0.0111


In [5]:
# Ensembles on sarc_temporal
te = pd.read_csv(F / "ensemble_sarc_temporal/ensemble_results.csv")
U.show(te[["method", "n_members", "val_auc_oof", "test_roc_auc", "test_f1", "test_p75_recall", "d_auc_vs_best_single", "ci_low", "ci_high"]])

,method,n_members,val_auc_oof,test_roc_auc,test_f1,test_p75_recall,d_auc_vs_best_single,ci_low,ci_high
0,best_single,13,0.8160,0.8126,0.7087,0.7132,0.0000,0.0000,0.0000
1,soft_vote,13,0.8155,0.8126,0.7081,0.7133,-0.0000,-0.0006,0.0005
2,greedy_select,13,0.8249,0.8211,0.7146,0.7306,0.0085,0.0079,0.0090
3,stack_lr,13,0.8270,0.8235,0.7178,0.7373,0.0108,0.0103,0.0114
4,stack_lr_l1,13,0.8270,0.8235,0.7177,0.7373,0.0108,0.0103,0.0114
5,shapley_stack,13,0.8270,0.8235,0.7178,0.7373,0.0108,0.0103,0.0114
6,stack_lgbm_meta,13,0.8273,0.8239,0.7186,0.7369,0.0113,0.0108,0.0118
7,blend_meta_lr,13,0.8275,0.8240,0.7184,0.7375,0.0114,0.0108,0.0119
8,blend_meta_lr+mbr_expected_f1,13,NaN,0.8240,0.7292,NaN,NaN,NaN,NaN
9,blend_meta_lr+mbr_expected_prec75,13,NaN,0.8240,0.7229,NaN,NaN,NaN,NaN


## 10.4 Tuning the from-scratch Transformer

In [6]:
# Configurations against the ESIM
rows = []
for f in sorted(glob.glob(str(U.EXP / "12_scratch_nn/results/scratch_transformer_pair*__sarc_random.json"))):
    j = U.read_json(f); a = j.get("args", {})
    rows.append({"run": __import__('pathlib').Path(f).name.split("__")[0], "params": j["params"], "lr": a.get("lr", 3e-4), "layers": a.get("tf_layers", 4),
                 "dim": a.get("tf_dim", 192), "dropout": a.get("tf_drop", 0.1), "batch": a.get("bs", 256),
                 "best_epoch": j["best_epoch"], "val_auc": j["val_auc"], "test_auc": j["sarc_random"]["roc_auc"]})
j = U.read_json(U.EXP / "12_scratch_nn/results/scratch_pair__sarc_random.json")
rows.append({"run": "ESIM (reference)", "params": j["params"], "best_epoch": j["best_epoch"], "val_auc": j["val_auc"], "test_auc": j["sarc_random"]["roc_auc"]})
U.show(pd.DataFrame(rows))

,run,params,lr,layers,dim,dropout,batch,best_epoch,val_auc,test_auc
0,scratch_transformer_pair,7645825,0.0003,4.0,192.0,0.1,256.0,6,0.8013,0.8052
1,scratch_transformer_pair_cfg2,7645825,0.0010,4.0,192.0,0.1,512.0,5,0.8011,0.8056
2,scratch_transformer_pair_cfg3,9434113,0.0003,2.0,256.0,0.1,256.0,6,0.8016,0.8053
3,scratch_transformer_pair_cfg4,7645825,0.0003,4.0,192.0,0.2,256.0,7,0.8038,0.8067
4,ESIM (reference),7788481,NaN,NaN,NaN,NaN,NaN,7,0.8204,0.8242


## 10.5 How replies end on other datasets
TweetEval and FigLang are human-annotated; news headlines are labelled by source. Only SARC labels come from a stripped `/s`.

In [7]:
# Last character by label and AUC of 'no full stop'
e = pd.read_csv(F / "endings_by_dataset.csv")
U.show(e[["set", "rows", "auc_no_full_stop", "full stop | not sarcastic", "full stop | sarcastic", "exclamation | not sarcastic", "exclamation | sarcastic"]])

,set,rows,auc_no_full_stop,full stop | not sarcastic,full stop | sarcastic,exclamation | not sarcastic,exclamation | sarcastic
0,sarc_random,137065,0.5590,0.5246,0.4066,0.0458,0.1349
1,figlang_reddit,6144,0.5964,0.5622,0.3694,0.0409,0.1431
2,figlang_twitter,6454,0.4275,0.2225,0.3676,0.0594,0.0614
3,tweeteval_irony,4583,0.5054,0.2274,0.2165,0.0547,0.0729
4,news_headlines,28375,0.5064,0.0157,0.0028,0.0068,0.0009


## 10.6 The team ensemble notebook
`modelling/Ensemble/stacking&voting.ipynb` combines LightGBM, XGBoost, LR and SVM but computes AUC from 0/1 predictions. The same four members with the step-7 protocol:

In [8]:
# Same four members, AUC on scores
t = U.read_json(F / "team_stack_comparison.json")
display(pd.Series({k: v for k, v in t.items() if isinstance(v, float)}).round(4))
b = t["stack_vs_lgbm_baseline"]; print(f"stack vs LightGBM baseline: {b['diff']:+.4f} [{b['ci_low']:+.4f}, {b['ci_high']:+.4f}]")

soft_vote_test_auc                  0.8137
hard_vote_test_auc_on_vote_share    0.7702
stack_lr_val_auc_oof                0.8114
stack_lr_test_auc                   0.8142
stack_lr_test_auc_on_0_1_labels     0.7374
dtype: float64

stack vs LightGBM baseline: +0.0068 [+0.0062, +0.0073]


## Takeaways

- The submitted ensemble reproduces exactly from the scripts (largest difference 3e-5).
- Seeds barely move anything: the stack is 0.8417 ± 0.0003 over three seeds, far below every gain reported in steps 3–9.
- On the time split the improvements hold: best compliant LightGBM +0.021 over the baseline, ESIM and EBM about +0.013, and the 13-member stack 0.8240 (+0.032), so the conclusions are not specific to a random split.
- Tuning the from-scratch Transformer does not close its gap to the ESIM, which confirms that the RoBERTa gap is pretraining.
- The "no full stop means sarcastic" pattern appears in Reddit data (SARC and FigLang Reddit) but not in human-annotated TweetEval, and reverses on FigLang Twitter: it is a Reddit-specific signal, which supports flagging it as a possible artefact.
- With AUC computed on scores, the team's four-model stack reaches 0.8142 (+0.0068 over the baseline); the notebook's 0.71–0.74 comes from scoring 0/1 labels.